# 🚀 RadarMap Tactical Squad Stress Test Simulator

A dedicated multi-player concurrent stress test simulator and visual cockpit for **RadarMap**.

### 🎯 Key Capabilities
- **Simultaneous 10-Player Simulation**: Runs the entire squad concurrently in a single process.
- **Airsoft Bound Movement**: Realistic 15m–40m distance legs with heading lock and tactical stops.
- **Speed Regulation**: Dynamically maintains each player average speed over time with realistic pauses.
- **Network Quality Simulation (1–5)**: Intermittent disconnects with Ephemeral Latest-Only telemetry coalescing.
- **Tactical Marker Generation**: Drops tactical orders and indicators on Firebase whenever players stop.
- **Firebase Admin SDK**: Uses native persistent connection (no REST polling) with credentials in `notebooks/credentials/`.
- **10-Minute Time Limit**: Default 600s duration limit with live progress bar and automatic teardown.
- **Interactive PPI Vector Radar UI**: Real-time SVG radar screen and live squad telemetry grid.

## ⚙️ 1. Squad Roster & Simulation Configuration

Configure starting coordinates, room security, 10-minute time limit, and the simulated player table.
Table schema: `[callsign: str, avg_speed_mps: float, network_quality: int (1-5)]`

In [1]:
# =============================================================================
# 🎮 10-PLAYER SQUAD SETUP & ROOM CONFIGURATION
# =============================================================================
PLAYERS_TABLE = [
    ["VIPER-1",  4.5, 5],   # Point Scout: Fast runner, rock-solid network (★★★★★)
    ["VIPER-2",  3.0, 5],   # Squad Leader: Tactical pacing, rock-solid network (★★★★★)
    ["GHOST-1",  1.8, 4],   # Recon / Sniper: Slow stalker, good cellular (★★★★☆)
    ["GHOST-2",  2.2, 4],   # Spotter: Moderate pace, good cellular (★★★★☆)
    ["COBRA-1",  5.0, 3],   # Assault: High-speed bounding, moderate link (★★★☆☆)
    ["COBRA-2",  3.5, 3],   # Support: Sustained pace, occasional drops (★★★☆☆)
    ["COBRA-3",  2.8, 3],   # Breacher: Medium pace, occasional drops (★★★☆☆)
    ["EAGLE-1",  4.0, 2],   # Flanker 1: Fast movement, poor edge cellular (★★☆☆☆)
    ["EAGLE-2",  3.2, 2],   # Flanker 2: Medium movement, high packet loss (★★☆☆☆)
    ["WOLF-1",   5.5, 1],   # Rear Guard: Fast runner, severe degradation (★☆☆☆☆)
]

# Squad Room & PIN Security
ROOM_NAME = "STRESS"           # Room name / Squad ID (e.g. "STRESS" or "ALPHA")
PIN = "7788"                 # Mandatory PIN (4-16 alphanumeric chars)

# GPS Starting Center Coordinates (Default: Cupertino / SF tactical anchor)
LATITUDE = 37.33233141         # Center anchor latitude
LONGITUDE = -122.0312186       # Center anchor longitude

# Simulation Duration Limit (Default: 10 minutes = 600.0s)
SIMULATION_DURATION_SEC = 600.0  # Set to None for unlimited

# Tactical Movement Parameters
MIN_LEG_DISTANCE = 15.0        # Minimum distance (m) before direction change (airsoft bound)
MAX_LEG_DISTANCE = 40.0        # Maximum distance (m) before direction change (airsoft bound)
MARKER_DROP_PROBABILITY = 0.50 # Probability of dropping tactical marker upon stopping

# Network & Mode Settings
DRY_RUN = False                # Set to False to write live to Firebase Realtime Database
DATABASE_URL = "https://radarmap-8adf0-default-rtdb.firebaseio.com"
CREDENTIALS_PATH = "credentials/radarmap-8adf0-b9ba5edc8f6b.json"  # Service account in notebooks/credentials/
CLEANUP_ON_STOP = False         # False = keep nodes in Firebase RTDB so you can inspect them in Firebase Console

mode_str = "DRY RUN (offline physics & drops)" if DRY_RUN else "LIVE FIREBASE RTDB"
limit_str = f"{SIMULATION_DURATION_SEC:.0f}s ({SIMULATION_DURATION_SEC / 60.0:.1f} mins)" if SIMULATION_DURATION_SEC else "Unlimited"

print("✅ Stress Test Configuration Loaded:")
print(f"   • Room:        {ROOM_NAME} (PIN: {PIN})")
print(f"   • Center GPS:  ({LATITUDE:.6f}, {LONGITUDE:.6f})")
print(f"   • Squad Size:  {len(PLAYERS_TABLE)} concurrent simulated players")
print(f"   • Time Limit:  {limit_str}")
print(f"   • Bounds:      Leg Distance = {MIN_LEG_DISTANCE}m - {MAX_LEG_DISTANCE}m")
print(f"   • Mode:        {mode_str}")
print(f"   • Cleanup:     Purge upon stop = {CLEANUP_ON_STOP}")
print(f"   • Credentials: {CREDENTIALS_PATH}")
print()
print("📋 Seeded Squad Roster:")
print(f"   {'Callsign':<10} | {'Target Avg Speed':<18} | {'Network Quality'}")
print("   " + "-" * 52)
for cs, spd, q in PLAYERS_TABLE:
    stars = '★' * q + '☆' * (5 - q)
    print(f"   {cs:<10} | {spd:4.1f} m/s ({spd*3.6:4.1f} km/h) | {q}/5 {stars}")


✅ Stress Test Configuration Loaded:
   • Room:        STRESS (PIN: 7788)
   • Center GPS:  (37.332331, -122.031219)
   • Squad Size:  10 concurrent simulated players
   • Time Limit:  600s (10.0 mins)
   • Bounds:      Leg Distance = 15.0m - 40.0m
   • Mode:        LIVE FIREBASE RTDB
   • Cleanup:     Purge upon stop = False
   • Credentials: credentials/radarmap-8adf0-b9ba5edc8f6b.json

📋 Seeded Squad Roster:
   Callsign   | Target Avg Speed   | Network Quality
   ----------------------------------------------------
   VIPER-1    |  4.5 m/s (16.2 km/h) | 5/5 ★★★★★
   VIPER-2    |  3.0 m/s (10.8 km/h) | 5/5 ★★★★★
   GHOST-1    |  1.8 m/s ( 6.5 km/h) | 4/5 ★★★★☆
   GHOST-2    |  2.2 m/s ( 7.9 km/h) | 4/5 ★★★★☆
   COBRA-1    |  5.0 m/s (18.0 km/h) | 3/5 ★★★☆☆
   COBRA-2    |  3.5 m/s (12.6 km/h) | 3/5 ★★★☆☆
   COBRA-3    |  2.8 m/s (10.1 km/h) | 3/5 ★★★☆☆
   EAGLE-1    |  4.0 m/s (14.4 km/h) | 2/5 ★★☆☆☆
   EAGLE-2    |  3.2 m/s (11.5 km/h) | 2/5 ★★☆☆☆
   WOLF-1     |  5.5 m/s (19.8 km/h)

## 🛰️ 2. Interactive Tactical Radar & Stress Test UI

Click **Start Simulation** to launch all 10 players concurrently in a background worker thread.
Watch positions, speeds, stops, network drops, and tactical markers update in real time on the vector radar.

In [2]:
import os
import sys
import time
import math
import threading
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ensure current directory is on Python path
sys.path.insert(0, os.path.abspath("."))

from stress_test_simulator import StressTestCoordinator, PlayerSpec, MovementState

# UI Component definitions
ui_out = widgets.Output()
status_badge = widgets.HTML(value="<span style='color: #8E8E93; font-weight: bold;'>Status: READY</span>")
progress_bar = widgets.FloatProgress(
    value=0.0,
    min=0.0,
    max=SIMULATION_DURATION_SEC if SIMULATION_DURATION_SEC else 600.0,
    description='Progress:',
    bar_style='info',
    layout=widgets.Layout(width='280px')
)

btn_start = widgets.Button(description="▶ Start Simulation", button_style="success", tooltip="Launch 10-player stress test")
btn_stop = widgets.Button(description="⏹ Stop Simulation", button_style="danger", tooltip="Stop and clean up virtual squad room", disabled=True)

sim_thread = None
sim_coordinator = None

def generate_radar_html(state):
    players = state.get("players", [])
    elapsed = state.get("elapsed", 0.0)
    room_id = state.get("room_id", "")
    limit_sec = SIMULATION_DURATION_SEC if SIMULATION_DURATION_SEC else 600.0
    cx, cy = 190, 190
    scale = 0.55  # 1 meter = 0.55 pixels (350m perimeter fits in 380px frame)
    
    dots_svg = []
    rows_html = []
    
    for p in players:
        # Geodesic relative offsets in meters
        d_lat_m = (p.walker.current_lat - p.walker.center_lat) * 111139.0
        d_lon_m = (p.walker.current_lon - p.walker.center_lon) * (111139.0 * 0.795)
        
        svg_x = max(15, min(365, cx + (d_lon_m * scale)))
        svg_y = max(15, min(365, cy - (d_lat_m * scale)))
        
        status_color = "#00FF66" if p.net.is_connected else "#FF3B30"
        state_str = str(p.walker.state)
        state_color = "#FFCC00" if state_str == MovementState.STOPPED else "#00FFFF"
        
        # Player blip on radar
        dots_svg.append(f'<circle cx="{svg_x:.1f}" cy="{svg_y:.1f}" r="6" fill="{status_color}" stroke="{state_color}" stroke-width="2" />')
        dots_svg.append(f'<text x="{svg_x+8:.1f}" y="{svg_y+4:.1f}" fill="#E5E5EA" font-size="10" font-family="monospace" font-weight="bold">{p.callsign}</text>')
        
        # Heading vector line
        rad = math.radians(p.walker.current_heading_deg)
        vec_len = max(6.0, p.walker.current_speed_mps * 3.0)
        vec_x = svg_x + (vec_len * math.sin(rad))
        vec_y = svg_y - (vec_len * math.cos(rad))
        dots_svg.append(f'<line x1="{svg_x:.1f}" y1="{svg_y:.1f}" x2="{vec_x:.1f}" y2="{vec_y:.1f}" stroke="{state_color}" stroke-width="1.5" />')
        
        # Table row
        rows_html.append(f"""
        <tr style="border-bottom: 1px solid #2C2C2E;">
            <td style="padding: 5px 8px; font-weight: bold; color: #FFFFFF; font-family: monospace;">{p.callsign}</td>
            <td style="padding: 5px 8px; color: {status_color}; font-weight: bold;">{'🟢 ON' if p.net.is_connected else '🔴 OFF'}</td>
            <td style="padding: 5px 8px; color: #8E8E93;">{p.spec.network_quality}/5</td>
            <td style="padding: 5px 8px; color: {state_color}; font-weight: bold;">{state_str}</td>
            <td style="padding: 5px 8px; color: #30D158;">{p.walker.current_speed_mps:.1f} m/s</td>
            <td style="padding: 5px 8px; color: #BF5AF2;">{p.walker.cumulative_avg_speed_mps:.1f} / {p.spec.target_avg_speed_mps:.1f} m/s</td>
            <td style="padding: 5px 8px; color: #64D2FF;">{p.packets_succeeded} / {p.packets_coalesced_offline}</td>
            <td style="padding: 5px 8px; color: #FF9F0A; font-weight: bold;">{len(p.markers_placed)}</td>
        </tr>
        """)
        
    html = f"""
    <div style="background-color: #1C1C1E; color: #F2F2F7; border-radius: 12px; padding: 16px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; box-shadow: 0 6px 24px rgba(0,0,0,0.6); max-width: 980px; margin-top: 10px;">
        <!-- Header -->
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #3A3A3C; padding-bottom: 10px; margin-bottom: 14px;">
            <div>
                <span style="font-size: 18px; font-weight: bold; color: #00FF66;">🎯 RadarMap Tactical Squad Radar</span>
                <span style="background: #2C2C2E; color: #64D2FF; padding: 2px 8px; border-radius: 6px; font-size: 12px; margin-left: 10px; font-family: monospace;">ROOM: {room_id}</span>
                <span style="background: #2C2C2E; color: #FF9F0A; padding: 2px 8px; border-radius: 6px; font-size: 12px; margin-left: 6px; font-family: monospace;">FIREBASE ADMIN SDK</span>
            </div>
            <div style="font-size: 13px; font-family: monospace; color: #FFD60A;">
                ⏱️ {elapsed:5.1f}s / {limit_sec:.0f}s | 📡 {state.get('telemetry_sent', 0)} pkts | 🎯 {state.get('markers_placed', 0)} mkrs
            </div>
        </div>
        
        <!-- Main Grid: Radar + Squad Table -->
        <div style="display: flex; gap: 16px; flex-wrap: wrap;">
            <!-- Tactical PPI Radar Screen -->
            <div style="background: #000000; border: 2px solid #30D158; border-radius: 12px; width: 380px; height: 380px; position: relative; overflow: hidden; box-shadow: inset 0 0 25px rgba(0,255,102,0.2);">
                <svg width="380" height="380" style="position: absolute; top: 0; left: 0;">
                    <!-- Concentric Ranges (100m, 200m, 300m) -->
                    <circle cx="190" cy="190" r="165" fill="none" stroke="#1C3829" stroke-width="1.5" stroke-dasharray="4,4"/>
                    <circle cx="190" cy="190" r="110" fill="none" stroke="#1C3829" stroke-width="1.5" stroke-dasharray="4,4"/>
                    <circle cx="190" cy="190" r="55"  fill="none" stroke="#1C3829" stroke-width="1.5" stroke-dasharray="4,4"/>
                    <line x1="190" y1="0" x2="190" y2="380" stroke="#1C3829" stroke-width="1"/>
                    <line x1="0" y1="190" x2="380" y2="190" stroke="#1C3829" stroke-width="1"/>
                    <circle cx="190" cy="190" r="4" fill="#00FF66"/>
                    <text x="195" y="185" fill="#30D158" font-size="9" font-family="monospace">ORIGIN</text>
                    <text x="195" y="140" fill="#1C6839" font-size="8" font-family="monospace">100m</text>
                    <text x="195" y="85"  fill="#1C6839" font-size="8" font-family="monospace">200m</text>
                    <text x="195" y="30"  fill="#1C6839" font-size="8" font-family="monospace">300m</text>
                    {''.join(dots_svg)}
                </svg>
            </div>
            
            <!-- Live Squad Telemetry Table -->
            <div style="flex: 1; min-width: 460px; overflow-x: auto;">
                <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
                    <thead>
                        <tr style="background: #2C2C2E; color: #8E8E93; text-align: left;">
                            <th style="padding: 6px 8px;">CALLSIGN</th>
                            <th style="padding: 6px 8px;">NET</th>
                            <th style="padding: 6px 8px;">QUAL</th>
                            <th style="padding: 6px 8px;">STATE</th>
                            <th style="padding: 6px 8px;">SPEED</th>
                            <th style="padding: 6px 8px;">AVG/TGT</th>
                            <th style="padding: 6px 8px;">SENT/COAL</th>
                            <th style="padding: 6px 8px;">MKRS</th>
                        </tr>
                    </thead>
                    <tbody>
                        {''.join(rows_html)}
                    </tbody>
                </table>
            </div>
        </div>
    </div>
    """
    return html

def on_sim_tick(state):
    elapsed = state.get("elapsed", 0.0)
    progress_bar.value = min(progress_bar.max, elapsed)
    with ui_out:
        clear_output(wait=True)
        display(HTML(generate_radar_html(state)))

def start_simulation_clicked(b):
    global sim_thread, sim_coordinator
    btn_start.disabled = True
    btn_stop.disabled = False
    status_badge.value = "<span style='color: #30D158; font-weight: bold;'>🟢 Status: RUNNING</span>"
    progress_bar.value = 0.0
    
    sim_coordinator = StressTestCoordinator(
        center_lat=LATITUDE,
        center_lon=LONGITUDE,
        players=[PlayerSpec(c, s, q) for c, s, q in PLAYERS_TABLE],
        room_name=ROOM_NAME,
        pin=PIN,
        database_url=DATABASE_URL,
        credentials_path=CREDENTIALS_PATH,
        dry_run=DRY_RUN,
        min_leg_dist=MIN_LEG_DISTANCE,
        max_leg_dist=MAX_LEG_DISTANCE,
        marker_probability=MARKER_DROP_PROBABILITY,
        duration_sec=SIMULATION_DURATION_SEC,
        cleanup_on_stop=CLEANUP_ON_STOP,
        on_tick=on_sim_tick,
    )
    
    def run_worker():
        try:
            sim_coordinator.run()
        finally:
            btn_start.disabled = False
            btn_stop.disabled = True
            status_badge.value = "<span style='color: #FF9F0A; font-weight: bold;'>⏹ Status: COMPLETED / STOPPED</span>"
            
    sim_thread = threading.Thread(target=run_worker, daemon=True)
    sim_thread.start()

def stop_simulation_clicked(b):
    global sim_coordinator
    if sim_coordinator:
        sim_coordinator.is_running = False
    btn_start.disabled = False
    btn_stop.disabled = True
    status_badge.value = "<span style='color: #FF453A; font-weight: bold;'>⏹ Status: STOPPING...</span>"

btn_start.on_click(start_simulation_clicked)
btn_stop.on_click(stop_simulation_clicked)

# Render Controls & Container
control_bar = widgets.HBox([btn_start, btn_stop, status_badge, progress_bar], layout=widgets.Layout(align_items='center', gap='15px'))
display(control_bar)
display(ui_out)


Output()

## 🖥️ 3. Headless / Terminal Output Run

Alternatively, execute the simulation directly with the ANSI terminal dashboard:

In [3]:
# Run directly in terminal/cell output mode
coordinator = StressTestCoordinator(
    center_lat=LATITUDE,
    center_lon=LONGITUDE,
    players=[PlayerSpec(c, s, q) for c, s, q in PLAYERS_TABLE],
    room_name=ROOM_NAME,
    pin=PIN,
    database_url=DATABASE_URL,
    credentials_path=CREDENTIALS_PATH,
    dry_run=DRY_RUN,
    duration_sec=SIMULATION_DURATION_SEC,
    cleanup_on_stop=CLEANUP_ON_STOP,
)
coordinator.run()


[SIMULATOR] Running in --dry-run mode (local physics & network simulation, no Firebase upload).
[SIMULATOR] Initialized virtual room 'STRESSS5VHU48YRK' with 10 players.

🚀 RADARMAP TACTICAL STRESS SIMULATOR: 10 CONCURRENT SQUAD PLAYERS
📍 Center: (37.332331, -122.031219) | Room: STRESSS5VHU48YRK | Mode: DRY RUN (offline test)
⏱️ Time Limit: 600s (10.0 min)
📡 Bandwidth Scaling: Rate = 1.00 Hz | Interval = 1.00s | Heartbeat = 10.0s
Press Ctrl+C to safely terminate and clean up.


⏱️ [T+   0.0s] Room: STRESSS5VHU48YRK | Live Telemetry: 10 pkts | Delta Gated: 0 | Markers: 0
--------------------------------------------------------------------------------------------------------------
CALLSIGN   | STATUS  | NET   | STATE     | V_INST  | V_AVG/TGT   | DIST    | PKTS_OK | COALESCE | MARKERS
--------------------------------------------------------------------------------------------------------------
VIPER-1    | 🟢 ON  | 5/5   | RUNNING   |  6.0m/s |  6.0/4.5    |     0m  | 1       | 0        | 